In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json

from utils.llm import generate_structured
from utils.prompt_loader import load_prompt
from utils.case_schema import PatientCase

LLM PROJECT ROOT: C:\Users\seifa\OneDrive\Desktop\amd-hackathon\memoria-md
ENV EXISTS: True


In [3]:
with open(
    "../outputs/heart_failure_blueprint.json",
    "r",
    encoding="utf-8"
) as f:

    blueprint = json.load(f)

In [4]:
print(type(blueprint))

<class 'dict'>


In [5]:
print(blueprint.keys())

dict_keys(['topic', 'learning_objectives', 'core_concepts', 'prerequisite_concepts', 'definitions', 'clinical_features', 'risk_factors', 'history_questions', 'physical_exam', 'investigations', 'diagnosis', 'management', 'complications', 'red_flags', 'must_not_miss_concepts', 'common_student_mistakes', 'clinical_reasoning_points', 'assessment_opportunities', 'high_yield_facts'])


In [6]:
prompt = load_prompt(
    "case_generator.md",
    BLUEPRINT=json.dumps(
        blueprint,
        indent=2
    )
)

print(prompt[:1500])

You are MemoriaMD's Clinical Case Generator.

You are an expert medical educator and OSCE case designer.

Your responsibility is to convert a Teaching Blueprint into ONE realistic, internally consistent patient simulation suitable for senior medical students.

MISSION

Generate ONE complete PatientCase.

The generated patient should resemble a standardized patient in an OSCE examination.

The case should encourage clinical reasoning rather than pattern recognition.

SOURCE OF TRUTH

The Teaching Blueprint is the ONLY source of knowledge.

Use ONLY information contained in the Teaching Blueprint.

Do NOT introduce diseases, investigations, treatments, or medical facts that are absent from the blueprint.

If information is unavailable, leave the corresponding field empty rather than inventing details.

CASE DESIGN RULES

Generate ONE patient only.

Every section must be internally consistent.

History must support examination findings.

Examination findings must support investigations.



In [7]:
case = generate_structured(
    prompt,
    PatientCase
)


Trying models/gemini-3.1-flash-lite
✓ Success using models/gemini-3.1-flash-lite


In [8]:
print(type(case))
print(case.patient.name)
print(case.ground_truth.final_diagnosis)
print(case.history.opening_statement)

<class 'utils.case_schema.PatientCase'>
Arthur Miller
Chronic Heart Failure with reduced Ejection Fraction (HFrEF)
I've been feeling really short of breath lately, especially when I'm moving around, and my ankles have been swelling up.


In [9]:
import json

with open(
    "../outputs/heart_failure_case.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        case.model_dump(),
        f,
        indent=2,
        ensure_ascii=False
    )

In [10]:
from utils.case_schema import PatientCase

print(PatientCase.model_json_schema())

{'$defs': {'CaseMetadata': {'properties': {'case_id': {'title': 'Case Id', 'type': 'string'}, 'module': {'title': 'Module', 'type': 'string'}, 'title': {'title': 'Title', 'type': 'string'}, 'difficulty': {'title': 'Difficulty', 'type': 'string'}, 'estimated_time_minutes': {'title': 'Estimated Time Minutes', 'type': 'integer'}}, 'required': ['case_id', 'module', 'title', 'difficulty', 'estimated_time_minutes'], 'title': 'CaseMetadata', 'type': 'object'}, 'DisclosureRules': {'properties': {'volunteer_information': {'items': {'type': 'string'}, 'title': 'Volunteer Information', 'type': 'array'}, 'requires_direct_question': {'items': {'type': 'string'}, 'title': 'Requires Direct Question', 'type': 'array'}, 'never_disclose_unprompted': {'items': {'type': 'string'}, 'title': 'Never Disclose Unprompted', 'type': 'array'}}, 'required': ['volunteer_information', 'requires_direct_question', 'never_disclose_unprompted'], 'title': 'DisclosureRules', 'type': 'object'}, 'Examination': {'properties'